# From Dataset to Model: PyTorch Workflows
### Companion notebook - *From Pixels to Patients*, AAPM 2026 (Session 2 of 3)

Session 1 ended with an anonymized NIfTI dataset: CT images, RT structure masks, and metadata sidecars. This notebook turns that curated dataset into a reproducible PyTorch workflow:

1. Index the Session 1 NIfTI tree.
2. Split train/validation/test by patient hash to prevent leakage.
3. Build 3D PyTorch datasets and dataloaders.
4. Apply RT-aware augmentations that preserve mask geometry.
5. Train and evaluate a compact 3D U-Net for GTV segmentation.
6. Package weights, configuration, metrics, and split manifests for deployment review.

> This session assumes the Session 1 NIfTI export has already been generated. Training artifacts are written under `Session2/outputs/`, which is ignored by git.

---
## 0 - Setup

Install the runtime packages if needed, then import the reusable Session 2 code. Keep generated model artifacts under `Session2/outputs/` so the workshop repo stays clean.

In [ ]:
# Run once if your environment does not already have these packages.
# %pip install -q torch nibabel numpy pandas matplotlib scikit-learn

In [ ]:
from pathlib import Path
import sys

SESSION2 = Path.cwd()
if not (SESSION2 / "pixelstopatients").exists():
    SESSION2 = (Path.cwd() / "Session2").resolve()
sys.path.insert(0, str(SESSION2))

from pixelstopatients import NiftiSegmentationDataset, TinyUNet3D, TrainingConfig
from pixelstopatients.data import discover_cases, split_by_patient, write_index_csv
from pixelstopatients.engine import evaluate, seed_everything, train_one_epoch, write_json
from pixelstopatients.packaging import save_deployment_bundle

seed_everything(2026)
print("Session 2 folder:", SESSION2)

---
## 1 - Receive the Session 1 dataset

By default, this notebook looks for the Session 1 export at `aapm_nsclc/nifti`. Run Session 1 through the NIfTI export step before continuing here.

In [ ]:
NIFTI_ROOT = (SESSION2.parent / "aapm_nsclc" / "nifti").resolve()
if not NIFTI_ROOT.exists():
    raise FileNotFoundError(
        f"Session 1 NIfTI export not found at {NIFTI_ROOT}. "
        "Run Session 1 through the NIfTI export step first."
    )
print("Using Session 1 export:", NIFTI_ROOT)

---
## 2 - Index cases and split by patient

A patient can have multiple studies or series. Splitting by row can leak anatomy across train/validation/test; splitting by patient hash keeps the evaluation honest.

In [ ]:
config = TrainingConfig(
    nifti_root=NIFTI_ROOT,
    output_dir=SESSION2 / "outputs" / "runs" / "gtv_unet3d",
    epochs=2,
    batch_size=1,
)

records = discover_cases(config.nifti_root, required_masks=(config.target_mask,))
print(f"Found {len(records)} cases with target mask '{config.target_mask}'")
records[:3]

In [ ]:
splits = split_by_patient(records, config.val_fraction, config.test_fraction, config.seed)
for name, split_records in splits.items():
    patients = sorted({r.patient_id for r in split_records})
    print(f"{name:5s}: {len(split_records):3d} cases from {len(patients):3d} patients")
    write_index_csv(split_records, config.output_dir / f"{name}_index.csv")

---
## 3 - Build 3D datasets and dataloaders

The first channel is the CT image. Optional RT masks become context channels when available; missing context masks are filled with zeros so the model interface is stable across cases.

In [ ]:
from torch.utils.data import DataLoader

train_records = splits["train"]
val_records = splits["val"] or splits["train"]
test_records = splits["test"] or splits["val"] or splits["train"]

train_ds = NiftiSegmentationDataset(
    train_records,
    target_mask=config.target_mask,
    context_masks=config.context_masks,
    patch_size=config.patch_size,
    training=True,
    seed=config.seed,
)
val_ds = NiftiSegmentationDataset(val_records, config.target_mask, config.context_masks, config.patch_size)
test_ds = NiftiSegmentationDataset(test_records, config.target_mask, config.context_masks, config.patch_size)

sample = train_ds[0]
print("image tensor:", tuple(sample["image"].shape))
print("target tensor:", tuple(sample["target"].shape))
print("case:", sample["case_id"])

---
## 4 - Preview a training patch

This quick visual check catches the boring but expensive mistakes: empty masks, wrong orientation, mismatched crop sizes, or channels that do not line up.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

image = sample["image"][0].numpy()
target = sample["target"][0].numpy()
z = int(np.argmax(target.sum(axis=(1, 2)))) if target.any() else image.shape[0] // 2

plt.figure(figsize=(5, 5))
plt.imshow(image[z], cmap="gray")
plt.imshow(np.ma.masked_where(target[z] == 0, target[z]), cmap="autumn", alpha=0.45)
plt.title(f"{sample['case_id']} - z={z}")
plt.axis("off");

---
## 5 - Train and validate

For a live workshop, keep the model compact and epochs low. For real work, the same code path can scale to larger patches, more workers, longer schedules, and cross-validation.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=config.num_workers)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=config.num_workers)

model = TinyUNet3D(in_channels=1 + len(config.context_masks)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
print("device:", device)

In [ ]:
history = []
best_dice = -1.0
config.output_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, config.epochs + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device, threshold=config.threshold)
    row = {"epoch": epoch, "train": train_metrics, "val": val_metrics}
    history.append(row)
    print(row)
    if val_metrics["dice"] > best_dice:
        best_dice = val_metrics["dice"]
        torch.save(model.state_dict(), config.output_dir / "best_model_state.pt")

---
## 6 - Test and package for deployment review

The deployment bundle is deliberately boring: weights, config, metrics, split indexes, and a short model card. Session 3 can wrap this with input validation, output checks, drift monitoring, and governance.

In [ ]:
model.load_state_dict(torch.load(config.output_dir / "best_model_state.pt", map_location=device))
test_metrics = evaluate(model, test_loader, device, threshold=config.threshold)
metrics = {"history": history, "test": test_metrics, "n_cases": len(records)}
write_json(metrics, config.output_dir / "training_metrics.json")

bundle_dir = save_deployment_bundle(model, config, metrics, config.output_dir / "deployment_bundle")
print("test:", test_metrics)
print("deployment bundle:", bundle_dir)

---
## 7 - Where this goes next

Session 2 produces a model package that can be inspected and replayed. Session 3 should treat that package as an input to clinical deployment work: validating incoming studies, checking output sanity, monitoring drift, and defining governance around updates.